# Extract position of R\V Polarstern from CESM gridded data

Collocated CESM data is used in `figure-4-mosaic-time-series.ipynb` and `figures-A1-A2.ipynb`.

In [ ]:
import xarray as xr
import datetime as dt
from glob import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import utils

## Read CESM aerosol fields and Polarstern coordinates

In [3]:
def cesm_wrf_species(cesm):
    cesm["bc"] = 0.1216*cesm["bc_a4"]+0.1123*cesm["bc_a1"]
    cesm["bc"] += 0.7618*cesm["bc_a4"]+0.3783*cesm["bc_a1"]
    cesm["bc"] += 0.1164*cesm["bc_a4"]+0.0087*cesm["bc_a1"]
    cesm["bc"] += 0.0002*cesm["bc_a4"]+0.0000*cesm["bc_a1"]
    cesm["oc"] = 0.9886*cesm["soa_a2"]+0.1216*cesm["soa_a1"]+0.1123*cesm["pom_a1"]
    cesm["oc"] += 0.0114*cesm["soa_a2"]+0.7618*cesm["soa_a1"]+0.3783*cesm["pom_a1"]
    cesm["oc"] += 0.0000*cesm["soa_a2"]+0.1164*cesm["soa_a1"]+0.0087*cesm["pom_a1"]
    cesm["oc"] += 0.0000*cesm["soa_a2"]+0.0002*cesm["soa_a1"]+0.0000*cesm["pom_a1"]
    cesm["so4"] = 0.1216*cesm["so4_a1"]+0.0000*cesm["so4_a3"]+0.2376*cesm["so4_a2"]
    cesm["so4"] += 0.7618*cesm["so4_a1"]+0.0002*cesm["so4_a3"]+0.0001*cesm["so4_a2"]
    cesm["so4"] += 0.1164*cesm["so4_a1"]+0.0995*cesm["so4_a3"]+0.0000*cesm["so4_a2"]
    cesm["so4"] += 0.0002*cesm["so4_a1"]+0.9003*cesm["so4_a3"]+0.0000*cesm["so4_a2"]
    return cesm[["so4","oc","bc"]]

In [4]:
cesm = xr.open_mfdataset("/mnt/data/cesm/cesm-202004-aer.nc").sel(time=slice(dt.datetime(2020,4,1), dt.datetime(2020,4,30,23,59)))
cesm = cesm_wrf_species(cesm)
mosaic = xr.open_dataset("/mnt/data/mosaic/nav/mosaic_coords_2020-04.nc").sel(time=cesm.time)


## Perform extraction

In [ ]:
window = 3
cesm_xx, cesm_yy = np.meshgrid(cesm.lon.values, cesm.lat.values)

cesm_mosaic = []
for t in range(len(cesm.time)):
    lon, lat = mosaic.Longitude.values[t], mosaic.Latitude.values[t]
    j, i = utils.extract_point_from_grid(cesm_xx, cesm_yy, lon, lat)
    # make index arrays for window**2 nearest points, making sure 0 < i < nx
    nx, ny = len(cesm.lon), len(cesm.lat)
    r = window // 2
    imin, imax = max(0, i - r), min(nx, i + r + 1)
    jmin, jmax = max(0, j - r), min(ny, j + r + 1)
    islice = range(imin, imax)
    jslice = range(jmin, jmax)
    subset = cesm.isel(time=t, lat=jslice, lon=islice)
    subset = subset.assign_coords({
            'i': ('lon', [0,1,2]),
            'j': ('lat', [0,1,2]),
        }).swap_dims({'lon':'i', 'lat':'j'})
    cesm_mosaic.append(subset)
cesm_mosaic = xr.concat(cesm_mosaic, dim="time")

/tmp/ipykernel_14550/1937521023.py:21: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  cesm_mosaic = xr.concat(cesm_mosaic, dim="time")


In [8]:
cesm_mosaic.drop_vars(["lon","lat"]).to_netcdf("/mnt/data/cesm/cesm_mosaic.nc")